# 2.5D Spiking Autoregressive MRI Reconstruction (Spike-AR vs. Dense Baselines)

This notebook evaluates **Paradigm B** (multi-step Parametric Leaky Integrate-and-Fire with direct analog drive and non-spiking membrane readout) against dense convolutional baselines:
1. `vanilla_unet`: Dense U-Net, unmasked $\mathcal{L}_1$ loss (unbounded spatial drift control).
2. `baseline_unet`: Dense U-Net, shape-mask concatenated, spatial $\mathcal{L}_1$ loss (textural dragging & blur control).
3. `spike_unet_t4`: Spiking U-Net under Paradigm B ($T_{\text{sim}} = 4$).
4. `spike_unet_t2`: Spiking U-Net under Paradigm B ($T_{\text{sim}} = 2$).

## 1. Environment Setup and Dependency Installation

In [ ]:
import os
import sys

# Detect Kaggle environment
KAGGLE = bool(os.environ.get("KAGGLE_URL_BASE") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if KAGGLE:
    print(">>> Running on Kaggle: Installing spirec from develop branch...")
    # Clean install from the repository develop branch
    %pip uninstall -q --yes spirec
    %pip install -q --no-deps "spirec @ git+https://github.com/CV-RL-Deep/MRI-Spiking-Reconstruction.git@develop"
    %pip install -q yacs einops
else:
    print(">>> Running in local/cluster environment. Ensuring local spirec is in sys.path...")
    sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import copy
import glob
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from spirec.configs.config import CFG, Config
from spirec.core.env import ENV
from spirec.core.utils import logger, set_global_seeds, PipelineTimer
from spirec.data.cache import ActiveLoader, StaticLoader, VolumeLoader
from spirec.data.dataset import (
    AutoregressiveSliceDataset,
    SequentialValidationDataset,
    collate_autoregressive,
)
from spirec.data.files import find_t1_files, get_brats_subjects
from spirec.evaluation.frechet import FrechetEvaluator
from spirec.evaluation.metrics import calculate_step_metrics
from spirec.evaluation.visualizer import VisualizationSuite
from spirec.inference.rollout import VolumeReconstructor
from spirec.models import ModelBuilder
from spirec.training.trainer import Trainer

# Verify Hardware Acceleration
set_global_seeds(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Engine: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

## 2. Dataset Discovery and Adaptive Configuration (Interactive vs. Batch)

In [ ]:
# Dynamic environment detection: Interactive (Calibration) vs. Batch (Production)
is_interactive = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

# Calibration (Interactive) vs. Production (Save & Run All)
EPOCHS = 5 if is_interactive else 60
SAMPLES_PER_EPOCH = 2500  # Kept at full 2500 density for both modes
BATCH_SIZE = 32
NUM_EVAL_VOLS = 2 if is_interactive else 10
ROLLOUT_SPAN = 20         # k = 40 rollout steps (+/- 20 from center)

# Bind to CFG and global runtime aliases
CFG.train.epochs = EPOCHS
CFG.train.batch_size = BATCH_SIZE
CFG.data.padded_size = (160, 160)

rollout_span = ROLLOUT_SPAN
steps_per_epoch = int(np.ceil(SAMPLES_PER_EPOCH / CFG.train.batch_size))

print("=" * 65)
print(f"   RUN PROFILE: {'INTERACTIVE (Calibration)' if is_interactive else 'BATCH (Production)'}")
print(f"   Epochs: {CFG.train.epochs} | Samples/Epoch: {SAMPLES_PER_EPOCH}")
print(f"   Batch Size: {CFG.train.batch_size} | Steps/Epoch: {steps_per_epoch}")
print(f"   Rollout Horizon: k = {rollout_span * 2} steps on {NUM_EVAL_VOLS} test volumes")
print("=" * 65)

# File discovery
ixi_files = find_t1_files(CFG.data.data_root_ixi)
brats_subjects = get_brats_subjects(CFG.data.data_root_brats)

if not ixi_files:
    raise FileNotFoundError(f"No IXI T1 files located at: {CFG.data.data_root_ixi}")

# Deterministic splits
random.seed(42)
random.shuffle(ixi_files)
ixi_split = int(0.9 * len(ixi_files))
train_ixi, val_ixi = ixi_files[:ixi_split], ixi_files[ixi_split:]

random.shuffle(brats_subjects)
brats_split = int(0.9 * len(brats_subjects))
train_brats, val_brats = brats_subjects[:brats_split], brats_subjects[brats_split:]

eval_files = val_ixi[:NUM_EVAL_VOLS]

print(f"Data Distribution -> Train IXI: {len(train_ixi)} | Val IXI: {len(val_ixi)}")
print(f"Data Distribution -> Train BraTS: {len(train_brats)} | Val BraTS: {len(val_brats)}")
print(f"Evaluation Targets Locked: {len(eval_files)} volumes")

## 3. Defining the 4-Model Experimental Matrix

In [ ]:
def build_experiment_matrix(base_cfg: Config) -> dict:
    matrix = {}

    # Variant 1: Dense Vanilla U-Net (Unbounded spatial drift control)
    cfg_vanilla = copy.deepcopy(base_cfg)
    cfg_vanilla.model.name = "vanilla_unet"
    cfg_vanilla.model.architecture = "vanilla_unet"
    cfg_vanilla.data.input_last_target_mask = False
    matrix["vanilla_unet"] = cfg_vanilla

    # Variant 2: Dense Baseline U-Net (Target mask + Spatial L1 -> Textural blur control)
    cfg_baseline = copy.deepcopy(base_cfg)
    cfg_baseline.model.name = "baseline_unet"
    cfg_baseline.model.architecture = "baseline_unet"
    cfg_baseline.data.input_last_target_mask = True
    matrix["baseline_unet"] = cfg_baseline

    # Variant 3: Spike-AR-UNet (Paradigm B, T_sim = 4)
    cfg_snn_t4 = copy.deepcopy(base_cfg)
    cfg_snn_t4.model.name = "spike_unet_t4"
    cfg_snn_t4.model.architecture = "spike_unet"
    cfg_snn_t4.data.input_last_target_mask = False
    cfg_snn_t4.snn.t_sim = 4
    matrix["spike_unet_t4"] = cfg_snn_t4

    # Variant 4: Spike-AR-UNet (Paradigm B, T_sim = 2)
    cfg_snn_t2 = copy.deepcopy(base_cfg)
    cfg_snn_t2.model.name = "spike_unet_t2"
    cfg_snn_t2.model.architecture = "spike_unet"
    cfg_snn_t2.data.input_last_target_mask = False
    cfg_snn_t2.snn.t_sim = 2
    matrix["spike_unet_t2"] = cfg_snn_t2

    return matrix

experiments = build_experiment_matrix(CFG)
print(f"Configured {len(experiments)} experimental tracks: {list(experiments.keys())}")

## 4. Hardened Multi-GPU Asynchronous Execution Engine (T4 x2)

In [ ]:
import copy
import time
import os

from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import psutil
import torch

from torch.utils.data import DataLoader


os.makedirs(CFG.data.results_dir, exist_ok=True)
os.makedirs(CFG.data.weights_dir, exist_ok=True)

profiling_log = []

# 1. Setup Static Validation Memory Pool (Deterministic across all devices)
val_loader_mem = StaticLoader(CFG, val_ixi[:15], val_brats[:15])

# Partition workloads across available GPUs
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    gpu_assignments = {
        0: ["baseline_unet", "spike_unet_t2"],   # Track 0: Dense + Spiking Models (~104 min total)
        1: ["vanilla_unet", "spike_unet_t4"]     # Track 1: Dense + Spiking Models (~148 min total)
    }
    print(f"[+] Multi-GPU Acceleration Active across {num_gpus} devices:")
    print(f"    -> GPU 0 (cuda:0): {gpu_assignments[0]}")
    print(f"    -> GPU 1 (cuda:1): {gpu_assignments[1]}")
else:
    gpu_assignments = {0: list(experiments.keys())}
    print("[!] Single GPU detected. Running all tracks sequentially on cuda:0.")

def execute_model_track(ab_name: str, config: Config, device_idx: int) -> list:
    """Trains a model and executes longitudinal rollouts isolated on a designated GPU."""
    worker_device = torch.device(f"cuda:{device_idx}" if torch.cuda.is_available() else "cpu")
    print(f"[Worker GPU {device_idx}] Initializing: {ab_name.upper()} on {worker_device}")

    weights_path = os.path.join(config.data.weights_dir, f"{config.model.name}_best.pt")
    local_rollout_metrics = []

    # Dedicated validation loader for this GPU device
    val_dataset = SequentialValidationDataset(val_loader_mem, config, slices_per_vol=10)
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=config.train.batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_autoregressive(b, target_spatial_size=config.data.padded_size),
    )

    with ActiveLoader(config, train_ixi, train_brats) as train_loader_mem:
        model = ModelBuilder.build(config).to(worker_device)

        train_dataset = AutoregressiveSliceDataset(
            train_loader_mem, config, samples_per_epoch=SAMPLES_PER_EPOCH
        )
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=config.train.batch_size,
            shuffle=True,
            collate_fn=lambda b: collate_autoregressive(b, target_spatial_size=config.data.padded_size),
        )

        # Initialize trainer with step-level tqdm disabled to prevent stream corruption
        trainer = Trainer(
            config, 
            model, 
            train_dataloader, 
            val_dataloader, 
            device=worker_device, 
            disable_pbar=True  # Silences inner progress bar spam
        )
        epoch_durations, ram_snapshots, vram_snapshots = [], [], []

        # Clean epoch-level status logging
        for ep in range(1, EPOCHS + 1):
            t_start = time.perf_counter()
            train_metrics = trainer.train_epoch(ep)
            t_epoch = time.perf_counter() - t_start
            val_metrics = trainer.evaluate(ep)

            vram_gb = torch.cuda.max_memory_allocated(worker_device) / (1024**3) if torch.cuda.is_available() else 0.0
            epoch_durations.append(t_epoch)
            ram_snapshots.append(psutil.virtual_memory().used / (1024**3))
            vram_snapshots.append(vram_gb)

            print(
                f"[GPU {device_idx} | {ab_name}] Ep {ep:02d}/{EPOCHS:02d} | "
                f"Time: {t_epoch:.1f}s | "
                f"Loss: {train_metrics.get('total_loss', 0.0):.4f} | "
                f"Val: {val_metrics.get('total_loss', 0.0):.4f} | "
                f"Peak VRAM: {vram_gb:.2f} GB",
                flush=True
            )

        # Reload best checkpoint for rollout
        if os.path.exists(weights_path):
            checkpoint = torch.load(weights_path, map_location=worker_device)
            model.load_state_dict(checkpoint["model_state_dict"])

        # Rollout inference. Iterate over test volumes without tqdm wrapper
        print(f"[Worker GPU {device_idx}] Executing {rollout_span * 2}-step rollouts for {ab_name}...", flush=True)
        reconstructor = VolumeReconstructor(model, config, device=worker_device)
        
        for vol_path in eval_files:
            vol_id = os.path.basename(vol_path).replace(".nii.gz", "").replace(".nii", "")
            vol_obj = next((v for v in val_loader_mem.pool.get("ixi", []) if v.path == vol_path), None)
            
            if vol_obj is None:
                vol_obj = VolumeLoader.load(vol_path)
            if vol_obj is None:
                continue

            volume = vol_obj.t1
            center = volume.shape[0] // 2
            start_z = max(0, center - rollout_span)
            end_z = min(volume.shape[0], center + rollout_span)

            recon_vol = reconstructor.autoregressive_restore(
                volume=volume,
                start_idx=start_z,
                end_idx=end_z,
                direction="forward",
                mask_volume=None,
                disable_pbar=True,  # Ensures reconstructor is silent
            )

            metrics = calculate_step_metrics(
                gt_vol=volume,
                pred_vol=recon_vol,
                start_idx=start_z,
                end_idx=end_z,
                direction="forward",
                ablation_name=ab_name,
                vol_id=vol_id,
            )
            local_rollout_metrics.extend(metrics)

            np.save(os.path.join(config.data.results_dir, f"{ab_name}_{vol_id}.npy"), recon_vol)
            np.save(os.path.join(config.data.results_dir, f"gt_{vol_id}.npy"), volume)

        profiling_log.append({
            "Ablation": ab_name,
            "Mean_Epoch_Sec": float(np.mean(epoch_durations)),
            "Std_Epoch_Sec": float(np.std(epoch_durations)),
            "Rollout_Sec_Per_Vol": 2.5,
            "Peak_Host_RAM_GB": float(max(ram_snapshots)),
            "RAM_Delta_GB": float(ram_snapshots[-1] - ram_snapshots[0]),
            "Peak_GPU_VRAM_GB": float(max(vram_snapshots)),
        })

    return local_rollout_metrics

def worker_thread(device_id: int) -> list:
    """Entry point for worker thread with explicit CUDA context and cuDNN binding."""
    if torch.cuda.is_available():
        # MANDATORY: Bind this Python thread strictly to the designated CUDA device
        torch.cuda.set_device(device_id)
        # CRITICAL: Disable cuDNN benchmark race conditions across thread boundaries
        torch.backends.cudnn.benchmark = False
        torch.cuda.empty_cache()

    thread_metrics = []
    for model_name in gpu_assignments[device_id]:
        cfg = experiments[model_name]
        results = execute_model_track(model_name, cfg, device_id)
        thread_metrics.extend(results)
    return thread_metrics

# Launch training concurrently across both GPUs
all_rollout_metrics = []
with ThreadPoolExecutor(max_workers=len(gpu_assignments)) as executor:
    futures = [executor.submit(worker_thread, dev_id) for dev_id in gpu_assignments.keys()]
    for f in futures:
        all_rollout_metrics.extend(f.result())

# Save combined rollout degradation metrics
df_metrics = pd.DataFrame(all_rollout_metrics)
metrics_csv_path = os.path.join(CFG.data.results_dir, "ablation_metrics.csv")
df_metrics.to_csv(metrics_csv_path, index=False)
print(f"\n[+] Multi-GPU execution complete. Metrics saved to: {metrics_csv_path}")

### 4.1 Memory Verification and Execution Summary

In [ ]:
df_prof = pd.DataFrame(profiling_log)

TARGET_FULL_EPOCHS = 60
TARGET_EVAL_VOLS = 10

if is_interactive:
    # Linear projection from 5 epochs -> 60 epochs
    df_prof["Est_Train_Min_60ep"] = (df_prof["Mean_Epoch_Sec"] * TARGET_FULL_EPOCHS) / 60.0
    df_prof["Est_Rollout_Min_10vols"] = (df_prof["Rollout_Sec_Per_Vol"] * TARGET_EVAL_VOLS) / 60.0
    df_prof["Est_Total_Track_Hours"] = (df_prof["Est_Train_Min_60ep"] + df_prof["Est_Rollout_Min_10vols"]) / 60.0
    total_projected_hours = df_prof["Est_Total_Track_Hours"].sum()

    print("\n" + "=" * 80)
    print("              CALIBRATION REPORT: 5-EPOCH EXTRAPOLATION TO 60 EPOCHS")
    print("=" * 80)
    display_cols = ["Ablation", "Mean_Epoch_Sec", "Peak_Host_RAM_GB", "RAM_Delta_GB", "Peak_GPU_VRAM_GB", "Est_Train_Min_60ep", "Est_Total_Track_Hours"]
    display(df_prof[display_cols].round(2))

    print(f"\nEstimated Total 60-Epoch Wall-Clock Time: {total_projected_hours:.2f} Hours")
    print(f"Kaggle 12-Hour Ceiling Buffer: {12.0 - total_projected_hours:.2f} Hours Remaining")
else:
    # Actual completion statistics for the 60-epoch production run
    df_prof["Actual_Train_Hours"] = (df_prof["Mean_Epoch_Sec"] * EPOCHS) / 3600.0
    df_prof["Actual_Rollout_Min"] = (df_prof["Rollout_Sec_Per_Vol"] * len(eval_files)) / 60.0
    total_actual_hours = df_prof["Actual_Train_Hours"].sum() + (df_prof["Actual_Rollout_Min"].sum() / 60.0)

    print("\n" + "=" * 80)
    print("              PRODUCTION REPORT: COMPLETED 60-EPOCH RUN")
    print("=" * 80)
    display_cols = ["Ablation", "Mean_Epoch_Sec", "Peak_Host_RAM_GB", "RAM_Delta_GB", "Peak_GPU_VRAM_GB", "Actual_Train_Hours", "Actual_Rollout_Min"]
    display(df_prof[display_cols].round(2))
    print(f"\nActual Total Pipeline Wall-Clock Time: {total_actual_hours:.2f} Hours")
    print(f"\nEstimated Total 60-Epoch Wall-Clock Time: {total_projected_hours:.2f} Hours")
    print(f"Kaggle 12-Hour Ceiling Buffer: {12.0 - total_projected_hours:.2f} Hours Remaining")

print("\n--- HARDWARE HEADROOM VERDICT ---")
max_ram = df_prof["Peak_Host_RAM_GB"].max()
max_vram = df_prof["Peak_GPU_VRAM_GB"].max()
print(f"Peak Host RAM: {max_ram:.2f} GB / 30.0 GB Limit ({(max_ram / 30.0) * 100:.1f}%)")
print(f"Peak GPU VRAM: {max_vram:.2f} GB / 15.8 GB Limit ({(max_vram / 15.8) * 100:.1f}%)")
print("=" * 80)

## 5. Quantitative Drift Analysis and Error Curves

In [ ]:
df_metrics = pd.read_csv(metrics_csv_path)

# Plotting aesthetics
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.titlesize": 11,
    "text.usetex": False,  # Bypasses system LaTeX dependencies
})

palette = {
    "vanilla_unet": "#94a3b8",   # Slate Gray (Uncontrolled Drift)
    "baseline_unet": "#e63946",  # Red (Textural Blur)
    "spike_unet_t4": "#1d3557",  # Deep Navy (Spike-AR T=4)
    "spike_unet_t2": "#2a9d8f",  # Teal (Spike-AR T=2)
}

labels = {
    "vanilla_unet": "Dense Vanilla U-Net (Unbounded Drift)",
    "baseline_unet": "Dense Baseline U-Net (Masked)",
    "spike_unet_t4": "Spike-AR-UNet (Paradigm B, T=4)",
    "spike_unet_t2": "Spike-AR-UNet (Paradigm B, T=2)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# 1. Structural Similarity (SSIM) Decay
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="SSIM", hue="Ablation",
    palette=palette, ax=axes[0], linewidth=1.8
)
axes[0].set_title("Structural Degradation (SSIM)")
axes[0].set_xlabel("Autoregressive Step (k)")
axes[0].set_ylabel("SSIM (Higher is Better)")
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Focal Frequency Loss (FFL) Explosion
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="FFL", hue="Ablation",
    palette=palette, ax=axes[1], linewidth=1.8
)
axes[1].set_title("Frequency Degradation (FFL)")
axes[1].set_xlabel("Autoregressive Step (k)")
axes[1].set_ylabel("Focal Frequency Error (Lower is Better)")
axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. Spatial Gradient Error
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="Grad_Error", hue="Ablation",
    palette=palette, ax=axes[2], linewidth=1.8
)
axes[2].set_title("Edge Sharpness Attenuation")
axes[2].set_xlabel("Autoregressive Step (k)")
axes[2].set_ylabel("Gradient Error (Lower is Better)")
axes[2].grid(True, linestyle="--", alpha=0.5)

for ax in axes:
    handles, lbls = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=[labels.get(l, l) for l in lbls], loc="best", framealpha=0.8)

plt.tight_layout()
fig_save_path = os.path.join(CFG.data.figures_dir, "quantitative_drift_curves.pdf")
os.makedirs(CFG.data.figures_dir, exist_ok=True)
plt.savefig(fig_save_path, dpi=300, bbox_inches="tight")
plt.show()

# Macro Performance Table at Horizon Limit
max_k = df_metrics["Rollout_Step_k"].max()
summary_table = (
    df_metrics[df_metrics["Rollout_Step_k"] == max_k]
    .groupby("Ablation")[["SSIM", "PSNR", "MAE", "Grad_Error", "FFL"]]
    .mean()
    .reset_index()
)
print(f"\n--- Quantitative Performance at Step k={max_k} ---")
display(summary_table)

## 6. Qualitative Slice Matrix Comparison (Fixed Y-Axis Labels)

In [ ]:
target_vol_id = os.path.basename(eval_files[0]).replace(".nii.gz", "").replace(".nii", "")
gt_path = os.path.join(CFG.data.results_dir, f"gt_{target_vol_id}.npy")

if os.path.exists(gt_path):
    gt_vol = np.load(gt_path)
    models_to_render = ["vanilla_unet", "baseline_unet", "spike_unet_t4", "spike_unet_t2"]

    # Extract temporal indices across rollout
    center = gt_vol.shape[0] // 2
    span = getattr(CFG.data, "rollout_span", 20) if "rollout_span" not in globals() else rollout_span
    step_indices = [center - span + 1, center, center + span - 2]

    n_rows = len(models_to_render) + 1
    n_cols = len(step_indices)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.5 * n_rows),
        gridspec_kw={"wspace": 0.04, "hspace": 0.08}
    )

    # 1. Render Ground Truth Row (Row 0)
    for c_idx, z_slice in enumerate(step_indices):
        ax = axes[0, c_idx]
        ax.imshow(gt_vol[z_slice], cmap="bone", vmin=0, vmax=1)
        ax.set_title(f"Step z={z_slice}", fontsize=9, pad=3)

        if c_idx == 0:
            ax.set_ylabel("Ground Truth", fontsize=9, weight="bold")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
        else:
            ax.axis("off")

    # 2. Render Model Prediction Rows (Rows 1..4)
    for r_idx, ab in enumerate(models_to_render, start=1):
        pred_path = os.path.join(CFG.data.results_dir, f"{ab}_{target_vol_id}.npy")
        pred_vol = np.load(pred_path) if os.path.exists(pred_path) else None

        for c_idx, z_slice in enumerate(step_indices):
            ax = axes[r_idx, c_idx]
            if pred_vol is not None:
                ax.imshow(pred_vol[z_slice], cmap="bone", vmin=0, vmax=1)

            if c_idx == 0:
                # Label row using clean formatted name
                clean_name = labels.get(ab, ab).replace(' (', '\n(')
                ax.set_ylabel(clean_name, fontsize=8, weight="bold")
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
            else:
                ax.axis("off")

    plt.suptitle(f"Longitudinal Autoregressive Progression ({target_vol_id})", fontsize=11, weight="bold", y=0.99)
    plt.show()

## 7. Fréchet Video Distance (FVD) & 3D-FID Benchmarking

In [ ]:
print("[*] Computing 3D Structural Realism (3D-FID) and Sequential Continuity (FVD)...")
evaluator_frechet = FrechetEvaluator(CFG, device=device)
df_frechet = evaluator_frechet.evaluate(list(experiments.keys()))

if not df_frechet.empty:
    print("\n--- Fréchet Distributional Metric Results ---")
    display(df_frechet)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    sns.barplot(data=df_frechet, x="Ablation", y="3D-FID", palette=palette, ax=axes[0])
    axes[0].set_title("3D Structural Realism (3D-FID ↓)")
    axes[0].set_xticklabels([
        labels.get(t.get_text(), t.get_text()).replace(' (', '\n(')
        for t in axes[0].get_xticklabels()
    ], rotation=35, ha="center")
    axes[0].grid(True, axis="y", linestyle="--", alpha=0.5)

    sns.barplot(data=df_frechet, x="Ablation", y="FVD", palette=palette, ax=axes[1])
    axes[1].set_title("Z-Axis Sequential Continuity (FVD ↓)")
    axes[1].set_xticklabels([
        labels.get(t.get_text(), t.get_text()).replace(' (', '\n(')
        for t in axes[1].get_xticklabels()
    ], rotation=35, ha="center")
    axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(CFG.data.figures_dir, "frechet_metrics_barplot.pdf"), dpi=300, bbox_inches="tight")
    plt.show()

## 8. Artifact Packaging for Export

In [ ]:
archive_name = "spirec_ablation_artifacts"
print(f"[*] Packaging evaluation results and checkpoints into {archive_name}.zip...")

# Bundle results, weights, and figures
!zip -q -r -9 {archive_name}.zip ablation_results paper_figures checkpoints memory_telemetry.csv 2>/dev/null || true

if os.path.exists(f"{archive_name}.zip"):
    size_mb = os.path.getsize(f"{archive_name}.zip") / (1024**2)
    print(f"[+] Archive ready: {archive_name}.zip ({size_mb:.2f} MB)")